In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/InitializationHourly.ipynb"

In [ ]:
df_part3 = df_part3.loc[(df_part3["Hour"]==15)]# | (df_part3["Hour"]==15)]

In [ ]:
df_part3 = df_part3[["Date", "sentiment", "sentiment(t-1)", "sentiment_ln_return", "Hour", "PressReleaseCount"]]

In [ ]:
import pandas as pd

def load_equities_web(symbol, date_from,date_to):
    import pandas_datareader.data as web
    raw_data = web.DataReader(symbol, 'yahoo', pd.to_datetime(date_from), pd.to_datetime(date_to))# pd.datetime.now())
    data = raw_data.stack(dropna=False)['Adj Close'].to_frame().reset_index().rename(columns = {'Symbols':'symbol', 'Date':'date', 'Adj Close':'value'}).sort_values(by = ['symbol', 'date'])
    return pd.pivot_table(data, columns = 'symbol', index = 'date', values ='value')


prices = load_equities_web(['SPY', '^GSPC', '^VIX'], date_from = '2020-06-01', date_to = '2021-09-09')

In [ ]:
df.ffill(inplace=True)

In [ ]:
df_part3["Hour"].loc[df_part3["Date"]=='2021-03-02']

In [ ]:
df["PressReleaseCount"].plot(figsize=(20,6))

In [ ]:
df["vix"].plot(figsize=(20,6))

In [ ]:
df_part3["sentiment_ln_return"].ffill(inplace=True)

In [ ]:
df_part3['2020-09-02', "sentiment"]

In [ ]:
def backtest_strategy(prices, symbol_trade, symbol_volatility, volatility_threshold, capital, symbol_benchmark, df_part3):
    
    ################ STRATEGY 1 ##################

    df_init   = (prices[symbol_trade]*0).to_frame().assign(cash = 0)
    df_update = (prices[symbol_trade]*0).to_frame().assign(cash = 0)
    df_end    = (prices[symbol_trade]*0).to_frame().assign(cash = 0)
    
    df_init.iloc[0, df_init.columns.get_loc('cash')] = capital
    df_end.iloc[0, df_end.columns.get_loc('cash')]   = capital
    
    calendar = pd.Series(prices.index).iloc[1:]
    

    for date in calendar:
        prev_date = df_init.index[df_init.index<date][-1]
        print(date)
        df_init.loc[date, :] = df_end.loc[prev_date, :]
        
        port_value = df_init.loc[date, symbol_trade] * prices.loc[date, symbol_trade] + df_init.loc[date, 'cash']
        Date = str(date.date())
        # print("Date="+Date)
        # print(Date)
        if (Date in df_part3.Date):
            if (df_part3.loc[Date, "sentiment"] >0): # volatility is high -> be fully in cash
                df_end.loc[date, symbol_trade] = 0
                df_end.loc[date, 'cash']       = port_value
            else: # volatility is low -> be in market position
                df_end.loc[date, symbol_trade] = port_value/prices.loc[date, symbol_trade]
                df_end.loc[date, 'cash'] = 0
        else:
            df_end.loc[date, symbol_trade] = 0
            df_end.loc[date, 'cash']       = port_value
        df_update.loc[date] = df_end.loc[date] - df_init.loc[date]

    portval = (df_end*prices.assign(cash = 1)[[symbol_trade, 'cash']]).sum(axis = 1).to_frame().rename(columns = {0:'strategy1'})
 
    # portval = pd.DataFrame()
    
    # portval = pd.merge(portval1, portval2,how='left',right_on="Date", left_on="Date")
    portval['benchmark'] = prices[symbol_benchmark]
    portval = portval/portval.iloc[0].values
    
    return portval
    

res = backtest_strategy(prices = prices, symbol_trade = 'SPY', symbol_volatility = '^VIX', volatility_threshold = 0, capital = 10000, symbol_benchmark = '^GSPC',df_part3=df_part3)
pp = res.plot(figsize=(20,8))
fig = pp.get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/images/aa.png")

In [ ]:
df["sentiment"].plot(figsize=(20,6))

In [ ]:
df_part3["sentiment_ewm"].plot(figsize=(20,8))

In [ ]:
aa = df_part3["PressReleaseCount"].plot(figsize=(20,8))
fig1 = aa.get_figure()
fig1.savefig("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/images/bb.png")

In [ ]:
pd.(df_part3["PressReleaseCount"])

In [ ]:
df_part3["polarity"] = df_part3["polarity"]*4
df_part3[["polarity", "sentiment_ln_return"]].ewm(10).mean().plot()

In [ ]:
df_part3.sentiment

In [ ]:
from plotly import tools
import plotly.plotly as py
import plotly.figure_factory as ff
import plotly.tools as tls
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

fig = tls.make_subplots(rows=2, cols=1, shared_xaxes=True)

colors = ['#ff4500', '#92a1cf', '#6E6E6E']
avgs = ['10_d_avg', '50_d_avg', '200_d_avg']
# for i,c in zip(range(n),color):
#    ax1.plot(x, y,c=c)

for col, c in zip(avgs, colors):
    fig.append_trace({'x': amzn_df.index, 'y': amzn_df[col], 'type': 'scatter', 'name': col, 'line': {'color': c}}, 1, 1)
for col in ['close']:
    fig.append_trace({'x': amzn_df.index, 'y': amzn_df[col], 'type': 'scatter', 'name': 'Closing Price', 'line':{'color': '#01DF3A'}}, 2, 1)
    
fig['layout'].update(height=800,title='Relationship between MAs <br> and Closing Price',
                    paper_bgcolor='#F2DFCE', plot_bgcolor='#F2DFCE')
    
iplot(fig, filename='pandas/mixed-type subplots')